In [10]:
!pip install --no-index \
--find-links=/kaggle/input/datasets/harshilmaks/unsloth-offline-wheels/wheels \
unsloth trl peft accelerate bitsandbytes transformers datasets pillow numpy torch torchvision xformers

Looking in links: /kaggle/input/datasets/harshilmaks/unsloth-offline-wheels/wheels
Processing /kaggle/input/datasets/harshilmaks/unsloth-offline-wheels/wheels/unsloth-2026.3.17-py3-none-any.whl
Processing /kaggle/input/datasets/harshilmaks/unsloth-offline-wheels/wheels/trl-0.24.0-py3-none-any.whl
Processing /kaggle/input/datasets/harshilmaks/unsloth-offline-wheels/wheels/bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl
Processing /kaggle/input/datasets/harshilmaks/unsloth-offline-wheels/wheels/xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl
Processing /kaggle/input/datasets/harshilmaks/unsloth-offline-wheels/wheels/unsloth_zoo-2026.3.6-py3-none-any.whl (from unsloth)
Processing /kaggle/input/datasets/harshilmaks/unsloth-offline-wheels/wheels/tyro-1.0.11-py3-none-any.whl (from unsloth)
Processing /kaggle/input/datasets/harshilmaks/unsloth-offline-wheels/wheels/datasets-4.3.0-py3-none-any.whl
Processing /kaggle/input/datasets/harshilmaks/unsloth-offline-wheels/wheels/hf_transfe

In [11]:
import os
import torch

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:256"

torch.backends.cuda.matmul.allow_tf32 = True

In [12]:
import json
from pathlib import Path
from datasets import Dataset, Image

DATA_ROOT = Path("/kaggle/input/datasets/harshilmaks/ghost-architect-data/data")
IMG_DIR = DATA_ROOT / "ui_screenshots"
JSON_PATH = DATA_ROOT / "dataset_merged.json"

with open(JSON_PATH, "r") as f:
    raw = json.load(f)

text_rows = []
vision_rows = []

for item in raw:
    instruction = str(item.get("instruction", "")).strip()
    output = str(item.get("output", "")).strip()
    text = f"### Instruction:\n{instruction}\n\n### Output:\n{output}"

    img_name = Path(str(item.get("image_path", ""))).name
    img_path = IMG_DIR / img_name if img_name else None

    if img_path and img_path.exists():
        vision_rows.append({
            "text": text,
            "image": str(img_path),
        })
    else:
        text_rows.append({
            "text": text,
        })

text_only = Dataset.from_list(text_rows)
vision_ds = Dataset.from_list(vision_rows).cast_column("image", Image())

print(f"Text-only samples: {len(text_only)}")
print(f"Vision samples: {len(vision_ds)}")
print(text_only[0])
print(vision_ds[0].keys())

Text-only samples: 5000
Vision samples: 287
{'text': '### Instruction:\nGenerate PostgreSQL schema from this UI\n\n### Output:\nCREATE TABLE users (\n    id SERIAL PRIMARY KEY,\n    profile_picture TEXT,\n    full_name VARCHAR(255),\n    email VARCHAR(255) UNIQUE NOT NULL,\n    member_since DATE,\n    phone_number VARCHAR(50)\n);'}
dict_keys(['text', 'image'])


In [ ]:
from datasets import Dataset

text_only = Dataset.from_list(text_only)
multimodal = Dataset.from_list(multimodal)

print(type(text_only))  # should be Dataset

In [13]:
# ENV FIX CELL (add this)
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [1]:
import torch
print(torch.cuda.memory_allocated() / 1024**3, "GB used")

0.0 GB used


In [1]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="unsloth/gemma-3-12b-it-bnb-4bit",
    local_dir="/kaggle/working/gemma3_model"
)

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

'/kaggle/working/gemma3_model'

In [2]:
!zip -r gemma3_model.zip gemma3_model

  adding: gemma3_model/ (stored 0%)
  adding: gemma3_model/.gitattributes (deflated 87%)
  adding: gemma3_model/config.json (deflated 63%)
  adding: gemma3_model/added_tokens.json (stored 0%)
  adding: gemma3_model/model.safetensors.index.json (deflated 97%)
  adding: gemma3_model/special_tokens_map.json (deflated 77%)
  adding: gemma3_model/.cache/ (stored 0%)
  adding: gemma3_model/.cache/huggingface/ (stored 0%)
  adding: gemma3_model/.cache/huggingface/download/ (stored 0%)
  adding: gemma3_model/.cache/huggingface/download/.gitattributes.metadata (deflated 24%)
  adding: gemma3_model/.cache/huggingface/download/tokenizer.model.metadata (deflated 29%)
  adding: gemma3_model/.cache/huggingface/download/model-00002-of-00002.safetensors.metadata (deflated 30%)
  adding: gemma3_model/.cache/huggingface/download/special_tokens_map.json.metadata (deflated 25%)
  adding: gemma3_model/.cache/huggingface/download/chat_template.json.metadata (deflated 23%)
  adding: gemma3_model/.cache/huggi

In [15]:
from unsloth import FastLanguageModel
import torch

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-12b-it-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=compute_dtype,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    lora_alpha=64,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing=False,
    use_rslora=True,
    use_dora=True,
    finetune_vision_layers=False,
)

model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/unsloth/gemma-3-12b-it-bnb-4bit/resolve/main/config.json
[huggingface_hub.utils._http|WARNING]'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/unsloth/gemma-3-12b-it-bnb-4bit/resolve/main/config.json
Retrying in 1s [Retry 1/5].
[huggingface_hub.utils._http|WARNING]Retrying in 1s [Retry 1/5].
'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/unsloth/gemma-3-12b-it-bnb-4bit/resolve/main/adapter_config.json
[huggingface_hub.utils._http|WARNING]'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/unsloth/gemma-3-12b-it-bnb-4bit/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].
[huggingface_hub.utils._http|WARNING]Retrying in 1s [Retry 1/5].


RuntimeError: Unsloth: No config file found - are you sure the `model_name` is correct?
If you're using a model on your local device, confirm if the folder location exists.
If you're using a HuggingFace online model, check if it exists.

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Total params: {total/1e9:.2f}B")
print(f"Trainable params: {trainable/1e6:.2f}M")
print(torch.cuda.get_device_name(0))
print(f"{torch.cuda.memory_allocated()/1024**3:.2f} GB used")

In [ ]:
def format_text(batch):
    outputs = []

    instructions = batch.get("instruction", [])
    responses = batch.get("output", [])

    for inst, out in zip(instructions, responses):
        inst = str(inst).strip()
        out = str(out).strip()

        text = f"""### Instruction:
{inst}

### Output:
{out}"""

        outputs.append(text)

    return outputs   # ✅ LIST of strings

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=text_only,
    formatting_func=format_text,   # ✅ FIXED
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        num_train_epochs=1,
        max_steps=625,
        logging_steps=50,
        logging_first_step=True,      # ✅ ADD
        output_dir="./phase1",
        save_strategy="steps",
        save_steps=100,
        remove_unused_columns=False,
        dataloader_num_workers=0,     # ✅ ADD (IMPORTANT)
        report_to="none",             # ✅ ADD
        fp16=False,
        bf16=False,
    ),
)

trainer.train()

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    lora_alpha=32,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    use_rslora=True,
    use_dora=True,
    finetune_vision_layers=True,  # Phase 2
)

In [ ]:
from PIL import Image

def format_vision(batch):
    outputs = []

    instructions = batch.get("instruction", [])
    responses = batch.get("output", [])
    img_paths = batch.get("resolved_image_path", [])

    for inst, out, img_path in zip(instructions, responses, img_paths):

        inst = str(inst).strip()
        out = str(out).strip()

        image = None
        if img_path:
            try:
                with Image.open(img_path) as img:
                    image = img.convert("RGB")
            except:
                image = None

        text = f"""### Instruction:
{inst}

### Output:
{out}"""

        outputs.append({
            "text": text,
            "images": [image] if image else []
        })

    return outputs   # ✅ LIST

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=multimodal,
    formatting_func=format_vision,   # ✅ FIXED
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=2,
        learning_rate=5e-5,
        num_train_epochs=2,
        max_steps=150,
        logging_steps=10,
        logging_first_step=True,      # ✅ ADD
        output_dir="./phase2",
        save_strategy="steps",
        save_steps=50,
        remove_unused_columns=False,
        dataloader_num_workers=0,     # ✅ ADD (VERY IMPORTANT)
        report_to="none",             # ✅ ADD
        fp16=False,
        bf16=False,
    ),
)

trainer.train()

In [ ]:
model.save_pretrained("./final_adapter")
tokenizer.save_pretrained("./final_adapter")

print("✅ Adapter saved")

In [ ]:
!cp -r ./final_adapter /kaggle/working/
!ls -lh /kaggle/working/final_adapter